# 4.2 How accurate are the extracted research claims?

The 2,673 research claims were extracted from 41 sources by an LLM in June 2026 (the claims pipeline, Phase 4). Their **quotes** were validated against the source text automatically; their **labels** were not (`review_status` is still "pending" for all of them). This notebook measures them on a random sample.

**Sample:** `evaluation/claim_label_audit.csv`, 50 claims drawn at random from all 2,673 (seed 2026, `scripts/analysis/make_claim_label_set.py`). For each, compared with its verbatim quote:
- `claim_correct`: the claim says what the quote says, and stands on its own;
- `impact_type_correct`, `direction_correct`, `entity_correct`: the three labels.

**Who judged:** Claude drafted the judgments with a note on every 0, and you accepted them (CHANGELOG 031).

**Scope:** these are the June 2026 claims. Claims added by Phase 8 need their own sample (ROADMAP 8.4).

In [1]:
import _setup
_setup.setup()

In [2]:
import pandas as pd
from scipy.stats import binomtest

CHECKS = ["claim_correct", "impact_type_correct", "direction_correct", "entity_correct"]
audit = pd.read_csv("evaluation/claim_label_audit.csv")
audit["all_four"] = audit[CHECKS].all(axis=1).astype(int)


def rate(column):
    k, n = int(audit[column].sum()), len(audit)
    interval = binomtest(k, n).proportion_ci(method="wilson")  # Wilson: sound at n = 50 and rates near 1
    return {"right": k, "of": n, "rate": round(k / n, 2), "95% interval": f"{interval.low:.2f}-{interval.high:.2f}"}


pd.DataFrame({column: rate(column) for column in CHECKS + ["all_four"]}).T

,right,of,rate,95% interval
claim_correct,40,50,0.8,0.67-0.89
impact_type_correct,41,50,0.82,0.69-0.90
direction_correct,38,50,0.76,0.63-0.86
entity_correct,42,50,0.84,0.71-0.92
all_four,20,50,0.4,0.28-0.54


## What goes wrong

Every claim with at least one 0, with the note that explains it.

In [3]:
wrong = audit[audit.all_four == 0]
pd.set_option("display.max_colwidth", 120)
wrong[["claim_text", "impact_type_clean", "impact_direction_clean"] + CHECKS + ["note"]].reset_index(drop=True)

,claim_text,impact_type_clean,impact_direction_clean,claim_correct,impact_type_correct,direction_correct,entity_correct,note
0,The AI tool bolstered workers’ daily performance and reduced the time they needed for learning activities.,productivity_effect,negative,1,1,0,1,direction should be positive
1,AI adoption leads to higher sales.,productivity_effect,positive,0,1,1,0,quote '31% higher sales' alone cannot support a causal claim; firms are not an industry
2,"Contracting gives companies access to skills through contractors, freelancers, or temporary workers.",skill_change,positive,1,0,1,1,not about AI; not a skill change
3,The future of good jobs depends in large part on our ability to harness new technology platforms such as artificial ...,job_creation,positive,1,1,0,1,"conditional statement: uncertain, not positive"
4,Responses reveal gendered scoring that aligns with occupational segregation.,automation_exposure,uncertain,0,0,1,0,"no context: whose responses? about bias, not automation"
5,"The highest ranked tasks at work were writing (38 percent), administrative tasks (27 percent), and interpreting/tran...",task_transformation,positive,1,1,0,1,"describes usage, no direction"
6,The current generation of AI technologies is predominantly task replacing and generates only modest productivity gains.,productivity_effect,mixed,1,1,1,0,"entity should be tasks or firms, not 'non-AI hiring'"
7,Usage splits nearly evenly between automation (43%) and augmentation (57%) of human capabilities.,automation_risk,mixed,1,1,1,0,"entity is AI usage, not 'human capabilities'"
8,Employment growth for young workers has been stagnant.,employment_effect,negative,0,1,1,1,'stagnant' overstates 'some leveling off relative to other age groups'
9,Automation can easily complete these tasks and is continually adapting to perform more complicated duties.,automation_risk,negative,0,1,1,1,'these tasks' unresolved in the claim


## Which of these reach students?

In [4]:
shown = pd.read_csv("data/research/career_research_claims.csv", dtype=str)
in_app = audit[audit.claim_id.isin(shown.claim_id)]
print(f"claims the app shows: {shown.claim_id.nunique()} distinct, across {shown.onetsoc_code.nunique()} careers")
print(f"audited claims among them: {len(in_app)} of {len(audit)}; claim text right: {in_app.claim_correct.sum()} of {len(in_app)}")
in_app.loc[in_app.claim_correct == 0, ["claim_text", "note"]].reset_index(drop=True)

claims the app shows: 462 distinct, across 771 careers
audited claims among them: 14 of 50; claim text right: 9 of 14


,claim_text,note
0,Responses reveal gendered scoring that aligns with occupational segregation.,"no context: whose responses? about bias, not automation"
1,Automation can easily complete these tasks and is continually adapting to perform more complicated duties.,'these tasks' unresolved in the claim
2,"The sector could experience a decline in demand for basic cognitive skills, physical and manual skills, and higher c...",'the sector' unresolved
3,The companies that are achieving the largest productivity gains are redesigning how they operate.,claim says something the quote does not
4,The presence of a premium also reinforces that AI capabilities are valued.,'the premium' and sector unresolved; direction positive


## Findings

1. **Claim texts are mostly right overall (4 in 5), but less so among the claims the app shows**: 9 of the 14 audited claims that reach students. The failures there are claims that **don't stand on their own** ("The sector could experience…", "…complete these tasks…") or say more than the quote. Showing the quote doesn't fix a missing context, because the quote lacks it too.
   - A possible fix: ask the relevance check (4.1c) to also reject claims that don't stand on their own. That is a prompt change and a full re-run (about $1.40), so it's yours to decide; noted in ROADMAP 4.2.
2. **The labels are weaker**: only about 2 in 5 claims are right on all four checks, and direction is the least reliable (e.g. "negative" for a quote about workers' performance improving). **So the app shows no impact type or direction**, only the claim, the quote and the source.
3. **A few claims aren't about AI** although they passed the extraction's AI filter (3 of 50), and one of them, about contracting, is shown by the app.

**Limits:** 50 claims give wide intervals (see the table), and 14 shown claims are too few for a reliable rate; the judgments are Claude's, accepted by you.